# Full Fine-Tuning and Catastrophic Forgetting

Companion notebook for the [Full Fine-Tuning](https://codemia.io/courses/llm_inside_and_out/full_fine_tuning) lesson on Codemia.

**What you will build**: a full fine-tune of GPT-2 small on binary sentiment (SST-2 style) with a language-modeling head. Throughout training you will monitor two numbers in parallel: classification accuracy on the task, and perplexity on held-out WikiText. You will see perplexity climb while accuracy also climbs -- the model is *forgetting* its general language ability as it specializes. You will run this at three learning rates to see how aggressive the tradeoff is.

**Time**: ~25-40 minutes on a T4

**GPU required**: yes

## Setup

In [ ]:
%pip install -q torch transformers datasets matplotlib

In [ ]:
import math, time, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

When you train a pretrained model on a new, narrow task, the weights shift to reduce the task loss -- but they were doing useful work on the original pretraining distribution. Those directions get clobbered. This is **catastrophic forgetting**. It is the central reason practitioners prefer LoRA, adapters, and prompt tuning over full fine-tuning: smaller update = less collateral damage.

Concretely we will fine-tune on sentiment with the LM head (no classification head). We frame the task as text completion: `"Review: <text>\nSentiment:"` with the model required to produce `" positive"` or `" negative"`. This keeps the architecture identical to pretraining and makes forgetting measurable -- we can take the same model and compute WikiText perplexity from the same head.

## Data: SST-2 sentiment + WikiText for the forgetting probe

In [ ]:
# GLUE SST-2 is tiny (~67k train rows). We take a small slice to keep training fast.
sst2 = load_dataset('glue', 'sst2')
print(sst2)
print("one example:")
print(sst2['train'][0])

In [ ]:
# We use a slice of wikitext as the forgetting probe. Raw English text the model was pretrained on.
wiki = load_dataset('wikitext', 'wikitext-2-raw-v1', split='validation')
# Filter out blank lines and very short ones.
wiki_texts = [t for t in wiki['text'] if len(t.strip()) > 50][:64]
print(f"wiki probe: {len(wiki_texts)} paragraphs, first one:")
print(wiki_texts[0][:300])

## Tokenizer and input formatting

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# GPT-2 has no pad token. Reuse EOS; fine since we'll mask pads in the loss.
tokenizer.pad_token = tokenizer.eos_token

# Token IDs for " positive" and " negative" -- note the leading space, GPT-2 tokenization is space-sensitive.
pos_id = tokenizer.encode(' positive')[0]
neg_id = tokenizer.encode(' negative')[0]
print(f"' positive' -> {pos_id}")
print(f"' negative' -> {neg_id}")

In [ ]:
def format_example(text, label=None):
    # Prompt format: text is the review, the model must emit ' positive' or ' negative' next.
    prompt = f"Review: {text.strip()}\nSentiment:"
    if label is None:
        return prompt
    target = ' positive' if label == 1 else ' negative'
    return prompt + target

def prepare_batch(examples, max_len=96):
    # We train by computing LM loss *only on the target token*, not on the prompt.
    # That way gradients aim at the sentiment prediction, not at reproducing the review.
    prompts = [format_example(e['sentence'], None) for e in examples]
    fulls = [format_example(e['sentence'], e['label']) for e in examples]

    enc_full = tokenizer(fulls, return_tensors='pt', padding=True, truncation=True, max_length=max_len)
    enc_prompt = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=max_len)
    input_ids = enc_full.input_ids.to(device)
    attention_mask = enc_full.attention_mask.to(device)

    # Loss mask: 1 on target tokens, 0 on prompt and padding. For SST-2 the target is a single token,
    # so we only need to predict the position right after the prompt ends.
    labels = input_ids.clone()
    for i in range(len(examples)):
        # Find the number of prompt tokens for row i (unpadded).
        prompt_len = int(enc_prompt.attention_mask[i].sum().item())
        labels[i, :prompt_len] = -100  # prompt positions: ignore
        # Padding positions: ignore too
        labels[i, attention_mask[i] == 0] = -100
    return input_ids, attention_mask, labels

## Eval helpers: sentiment accuracy and WikiText perplexity

In [ ]:
@torch.no_grad()
def eval_sst_accuracy(model, n=200):
    model.eval()
    correct = 0
    for i in range(n):
        ex = sst2['validation'][i]
        prompt = format_example(ex['sentence'], None)
        ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        # Logits over the next token only. Compare the score at pos_id vs neg_id.
        logits = model(ids).logits[0, -1]
        pred = 1 if logits[pos_id] > logits[neg_id] else 0
        correct += int(pred == ex['label'])
    model.train()
    return correct / n

@torch.no_grad()
def eval_wiki_perplexity(model):
    # Cross-entropy averaged over all the tokens in the probe, then exp.
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for text in wiki_texts:
        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(device)
        out = model(**enc, labels=enc.input_ids)
        # out.loss is averaged over non-padded tokens. Weight by token count so we can recombine.
        n_tokens = enc.input_ids.numel()
        total_loss += out.loss.item() * n_tokens
        total_tokens += n_tokens
    model.train()
    return math.exp(total_loss / total_tokens)

## Fine-tune function

In [ ]:
def run_finetune(lr, n_steps=200, batch_size=8, eval_every=40):
    # Fresh copy of the pretrained model per run so each LR starts from the same weights.
    model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    # Pre-fine-tuning baseline: should be ~50% accuracy (chance) and ~30 perplexity.
    acc0 = eval_sst_accuracy(model)
    ppl0 = eval_wiki_perplexity(model)
    print(f"[lr={lr:g}] baseline: sst acc {acc0:.3f}  wiki ppl {ppl0:.2f}")

    history = [(0, acc0, ppl0)]
    train_idx = list(range(len(sst2['train'])))
    import random
    random.shuffle(train_idx)
    cursor = 0

    for step in range(1, n_steps + 1):
        batch = [sst2['train'][train_idx[(cursor + j) % len(train_idx)]] for j in range(batch_size)]
        cursor += batch_size
        input_ids, attention_mask, labels = prepare_batch(batch)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        opt.zero_grad(set_to_none=True)
        out.loss.backward()
        opt.step()

        if step % eval_every == 0 or step == n_steps:
            acc = eval_sst_accuracy(model)
            ppl = eval_wiki_perplexity(model)
            history.append((step, acc, ppl))
            print(f"[lr={lr:g}] step {step:4d}  sst acc {acc:.3f}  wiki ppl {ppl:.2f}  "
                  f"train loss {out.loss.item():.3f}")
    return history

## Run three learning rates

Expect the 1e-4 run to reach the highest SST accuracy the fastest -- and the highest wiki perplexity. The 5e-6 run barely moves either metric. 1e-5 sits in between.

In [ ]:
histories = {}
for lr in [5e-6, 1e-5, 1e-4]:
    print(f"\n================ LR = {lr} ================")
    histories[lr] = run_finetune(lr)

## Plot: accuracy up, wiki perplexity up too = forgetting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {5e-6: 'C0', 1e-5: 'C1', 1e-4: 'C2'}

ax = axes[0]
for lr, hist in histories.items():
    steps = [h[0] for h in hist]
    accs = [h[1] for h in hist]
    ax.plot(steps, accs, 'o-', color=colors[lr], label=f'lr={lr:g}')
ax.set_xlabel('fine-tune step')
ax.set_ylabel('SST-2 validation accuracy')
ax.set_title('Task accuracy (want: up)')
ax.legend()

ax = axes[1]
for lr, hist in histories.items():
    steps = [h[0] for h in hist]
    ppls = [h[2] for h in hist]
    ax.plot(steps, ppls, 's-', color=colors[lr], label=f'lr={lr:g}')
ax.set_xlabel('fine-tune step')
ax.set_ylabel('WikiText perplexity')
ax.set_title('General LM ability (want: flat -- but you will see it rise)')
ax.legend()

plt.tight_layout()
plt.show()

## The Pareto frontier

Plot the two metrics against each other. Each point is one (lr, step) snapshot. The upper-left is the win zone (high accuracy, low forgetting). Watch how the 1e-4 runs shoot toward the right (forgetting) while the 5e-6 runs barely budge.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for lr, hist in histories.items():
    accs = [h[1] for h in hist]
    ppls = [h[2] for h in hist]
    ax.plot(ppls, accs, 'o-', color=colors[lr], label=f'lr={lr:g}', alpha=0.8)
ax.set_xlabel('WikiText perplexity (lower = remembers pretraining)')
ax.set_ylabel('SST-2 accuracy (higher = learned task)')
ax.set_title('Task learning vs. forgetting trade-off')
ax.legend()
plt.tight_layout()
plt.show()

## Qualitative check: what did the forgetful model do to its language modeling?

Generate from the most aggressively trained model (1e-4) and watch it over-commit to the sentiment output format. It will try to produce "positive" / "negative" regardless of the prompt -- classic forgetting.

In [ ]:
# Re-run the 1e-4 fine-tune and keep the final model so we can sample from it.
model_burned = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
opt = torch.optim.AdamW(model_burned.parameters(), lr=1e-4)
import random
random.seed(0)
train_idx = list(range(len(sst2['train'])))
random.shuffle(train_idx)
for step in range(200):
    batch = [sst2['train'][train_idx[(step * 8 + j) % len(train_idx)]] for j in range(8)]
    input_ids, attention_mask, labels = prepare_batch(batch)
    out = model_burned(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    opt.zero_grad(set_to_none=True); out.loss.backward(); opt.step()

model_fresh = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

def continue_text(m, prompt, max_new=40):
    m.eval()
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    out = m.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    m.train()
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = "The quick brown fox"
print("--- pretrained GPT-2 ---")
print(continue_text(model_fresh, prompt))
print()
print("--- fine-tuned at 1e-4 (burned) ---")
print(continue_text(model_burned, prompt))

## What to try next

- Try a much smaller learning rate like 1e-6 and see whether you can reach the same accuracy without any measurable forgetting.
- Add a **KL regularizer** to the loss: penalize divergence from the original model's token distribution on wiki samples. This is the crude version of what RLHF does.
- Compare with **LoRA** from `finetuning/lora_from_scratch.ipynb`: same task, same learning rate, observe how the wiki perplexity stays flat because the base weights are frozen.
- Monitor a per-layer weight delta (`param - original_param`).norm() to see which layers changed most during fine-tuning. Usually it's the final few.